# Prediction du Churn Bancaire — Demonstration

> **Auteur :** Emmanuel TSAGUE — Data Scientist / Data Analyst  
> **Données :** Simulées / Anonymisées — Portfolio pédagogique  
> **GitHub :** https://github.com/TSAGUE25


## 1. Imports

Chargement des librairies.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
try:
    from xgboost import XGBClassifier
    print('XGBoost disponible')
except:
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier
    print('Fallback vers GradientBoosting')
np.random.seed(42)

## 2. Dataset simule — 15 000 clients bancaires

16 % de taux de churn — desequilibre de classes.


In [ ]:
N = 15000
df = pd.DataFrame({
    'age':              np.random.randint(18, 75, N),
    'anciennete_ans':   np.random.randint(0, 20, N),
    'solde':            np.abs(np.random.normal(60000, 30000, N)),
    'nb_produits':      np.random.randint(1, 5, N),
    'membre_actif':     np.random.randint(0, 2, N),
    'salaire':          np.abs(np.random.normal(55000, 20000, N)),
    'credit_score':     np.random.randint(300, 850, N),
})
prob = 0.05 + 0.15*(df['membre_actif']==0) + 0.12*(df['nb_produits']==1) + np.random.uniform(-0.05, 0.05, N)
df['churn'] = (prob > 0.20).astype(int)
print(f'Taux de churn: {df.churn.mean():.1%}')

## 3. Pipeline XGBoost avec gestion du desequilibre

scale_pos_weight pour compenser le desequilibre 84/16.


In [ ]:
X = df.drop(columns='churn')
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

ratio = (y_train==0).sum() / (y_train==1).sum()
print(f'scale_pos_weight = {ratio:.1f}')

try:
    model = XGBClassifier(n_estimators=200, scale_pos_weight=ratio,
                          random_state=42, eval_metric='auc', verbosity=0)
except TypeError:
    model = XGBClassifier(n_estimators=200, random_state=42)

pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
pipe.fit(X_train, y_train)
y_proba = pipe.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_proba)
print(f'ROC-AUC: {auc:.4f}')
print(classification_report(y_test, pipe.predict(X_test), target_names=['No Churn','Churn']))

## 4. Distribution des scores par classe

Separation des probabilites entre churners et non-churners.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_proba[y_test==0], bins=40, alpha=0.6, color='steelblue', label='No Churn', density=True)
ax.hist(y_proba[y_test==1], bins=40, alpha=0.6, color='red',       label='Churn',    density=True)
ax.set_xlabel('Probabilite de churn')
ax.set_ylabel('Densite')
ax.set_title(f'Distribution des scores — AUC={auc:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/churn_score_distribution.png', dpi=120)
plt.show()